# 01 — Pipeline de Dados: Olist E-Commerce

**Objetivo:** Carregar, unir, limpar e transformar os 9 CSVs em um dataset master.

Etapas: Extração → Transformação → Limpeza → Carga

In [1]:
import pandas as pd
import numpy as np
import os

RAW_PATH       = '../data/raw/'
PROCESSED_PATH = '../data/processed/'
os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Arquivos disponíveis:")
for f in sorted(os.listdir(RAW_PATH)):
    print(f"  ✓ {f}")

Arquivos disponíveis:
  ✓ olist_customers_dataset.csv
  ✓ olist_geolocation_dataset.csv
  ✓ olist_order_items_dataset.csv
  ✓ olist_order_payments_dataset.csv
  ✓ olist_order_reviews_dataset.csv
  ✓ olist_orders_dataset.csv
  ✓ olist_products_dataset.csv
  ✓ olist_sellers_dataset.csv
  ✓ product_category_name_translation.csv


## Extração — Leitura dos CSVs

In [2]:
orders        = pd.read_csv(RAW_PATH + 'olist_orders_dataset.csv')
order_items   = pd.read_csv(RAW_PATH + 'olist_order_items_dataset.csv')
customers     = pd.read_csv(RAW_PATH + 'olist_customers_dataset.csv')
payments      = pd.read_csv(RAW_PATH + 'olist_order_payments_dataset.csv')
reviews       = pd.read_csv(RAW_PATH + 'olist_order_reviews_dataset.csv')
products      = pd.read_csv(RAW_PATH + 'olist_products_dataset.csv')
sellers       = pd.read_csv(RAW_PATH + 'olist_sellers_dataset.csv')
category_name = pd.read_csv(RAW_PATH + 'product_category_name_translation.csv')

datasets = {
    'orders': orders, 'order_items': order_items,
    'customers': customers, 'payments': payments,
    'reviews': reviews, 'products': products,
}
print(f"{'Tabela':<20} {'Linhas':>10} {'Colunas':>10}")
print("-" * 42)
for nome, df_temp in datasets.items():
    print(f"{nome:<20} {df_temp.shape[0]:>10,} {df_temp.shape[1]:>10}")

Tabela                   Linhas    Colunas
------------------------------------------
orders                   99,441          8
order_items             112,650          7
customers                99,441          5
payments                103,886          5
reviews                  99,224          7
products                 32,951          9


## Transformação — Converter datas e calcular métricas

In [3]:
colunas_data = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in colunas_data:
    orders[col] = pd.to_datetime(orders[col])

orders['delivery_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_purchase_timestamp']
).dt.days

orders['delay_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_estimated_delivery_date']
).dt.days

print("Métricas calculadas:")
print(orders[['delivery_days', 'delay_days']].describe().round(1))

Métricas calculadas:
       delivery_days  delay_days
count        96476.0     96476.0
mean            12.1       -11.9
std              9.6        10.2
min              0.0      -147.0
25%              6.0       -17.0
50%             10.0       -12.0
75%             15.0        -7.0
max            209.0       188.0


## Transformação — Join entre tabelas

In [4]:
payments_agg = (
    payments
    .groupby('order_id')
    .agg(
        payment_value=('payment_value', 'sum'),
        payment_installments=('payment_installments', 'max'),
        payment_type=('payment_type', lambda x: x.mode()[0])
    )
    .reset_index()
)

reviews_score = (
    reviews[['order_id', 'review_score']]
    .drop_duplicates('order_id')
)

products = products.merge(category_name, on='product_category_name', how='left')

df = (
    orders
    .merge(order_items,   on='order_id',    how='left')
    .merge(customers,     on='customer_id', how='left')
    .merge(payments_agg,  on='order_id',    how='left')
    .merge(reviews_score, on='order_id',    how='left')
    .merge(
        products[['product_id', 'product_category_name_english']],
        on='product_id', how='left'
    )
)

print(f"Dataset master: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
df.head(3)

Dataset master: 113,425 linhas × 25 colunas


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delay_days,...,freight_value,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_value,payment_installments,payment_type,review_score,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,...,8.72,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,1.0,voucher,4.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,...,22.76,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.0,boleto,4.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,...,19.22,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,3.0,credit_card,5.0,auto


## Limpeza — Filtro, nulos e outliers

In [5]:
df = df[df['order_status'] == 'delivered'].copy()
print(f"Após filtro 'delivered': {df.shape[0]:,} linhas")

df['delivery_days'] = df['delivery_days'].fillna(df['delivery_days'].median())
df['delay_days']    = df['delay_days'].fillna(0)
df['review_score']  = df['review_score'].fillna(df['review_score'].median())
df['product_category_name_english'] = (
    df['product_category_name_english'].fillna('unknown')
)

def remove_outliers(dataframe, coluna, multiplicador=3.0):
    Q1  = dataframe[coluna].quantile(0.25)
    Q3  = dataframe[coluna].quantile(0.75)
    IQR = Q3 - Q1
    antes = len(dataframe)
    limpo = dataframe[
        (dataframe[coluna] >= Q1 - multiplicador * IQR) &
        (dataframe[coluna] <= Q3 + multiplicador * IQR)
    ]
    print(f"  {coluna}: {antes - len(limpo)} outliers removidos")
    return limpo

print("\nRemoção de outliers:")
df = remove_outliers(df, 'payment_value')
df = remove_outliers(df, 'freight_value')
df = remove_outliers(df, 'delivery_days')

Após filtro 'delivered': 110,197 linhas

Remoção de outliers:
  payment_value: 4760 outliers removidos
  freight_value: 4989 outliers removidos
  delivery_days: 1203 outliers removidos


In [6]:
df.to_csv(PROCESSED_PATH + 'olist_master.csv', index=False)

print("✅ Pipeline concluído!")
print(f"   Arquivo: data/processed/olist_master.csv")
print(f"   Linhas:  {df.shape[0]:,}")
print(f"   Colunas: {df.shape[1]}")

✅ Pipeline concluído!
   Arquivo: data/processed/olist_master.csv
   Linhas:  99,245
   Colunas: 25
